# Data Science in Cybersecurity
## Data Cleaning and Preprocessing
### Task 2: Structural Error Correction
---

### Step 2.1 — Load the Dataset
We reload the raw dataset to continue from where Task 1 left off.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Load raw dataset
df = pd.read_csv('cybersecurity_raw.csv')

print(f'Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'\nOriginal column names:')
print(df.columns.tolist())

Dataset loaded: 520 rows × 17 columns

Original column names:
[' Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'Pkt Size Avg', 'Fwd IAT Mean', 'Bwd IAT Mean', 'Flow Bytes/s', 'Flow Pkts/s', 'Flag', 'Label']


---
### Step 2.2 — Standardize Column Names
Fix leading/trailing whitespace, convert to lowercase with underscores for consistency.

> **📸 SCREENSHOT HERE** — Capture the before and after column name comparison.

In [2]:
# Store original column names for comparison
original_cols = df.columns.tolist()

# Standardize: strip whitespace, lowercase, replace spaces/slashes with underscores
df.columns = (
    df.columns
    .str.strip()                  # remove leading/trailing whitespace
    .str.lower()                  # convert to lowercase
    .str.replace(' ', '_', regex=False)   # spaces to underscores
    .str.replace('/', '_per_', regex=False)  # slashes to _per_
    .str.replace('-', '_', regex=False)   # hyphens to underscores
)

new_cols = df.columns.tolist()

# Show before vs after
print('Column Name Changes:')
print(f'{"BEFORE":<25} {"AFTER"}')
print('-' * 50)
for old, new in zip(original_cols, new_cols):
    changed = '  ← CHANGED' if old != new else ''
    print(f'{repr(old):<25} {repr(new)}{changed}')

Column Name Changes:
BEFORE                    AFTER
--------------------------------------------------
' Flow ID'                'flow_id'  ← CHANGED
'Src IP'                  'src_ip'  ← CHANGED
'Src Port'                'src_port'  ← CHANGED
'Dst IP'                  'dst_ip'  ← CHANGED
'Dst Port'                'dst_port'  ← CHANGED
'Protocol'                'protocol'  ← CHANGED
'Timestamp'               'timestamp'  ← CHANGED
'Flow Duration'           'flow_duration'  ← CHANGED
'Tot Fwd Pkts'            'tot_fwd_pkts'  ← CHANGED
'Tot Bwd Pkts'            'tot_bwd_pkts'  ← CHANGED
'Pkt Size Avg'            'pkt_size_avg'  ← CHANGED
'Fwd IAT Mean'            'fwd_iat_mean'  ← CHANGED
'Bwd IAT Mean'            'bwd_iat_mean'  ← CHANGED
'Flow Bytes/s'            'flow_bytes_per_s'  ← CHANGED
'Flow Pkts/s'             'flow_pkts_per_s'  ← CHANGED
'Flag'                    'flag'  ← CHANGED
'Label'                   'label'  ← CHANGED


---
### Step 2.3 — Correct Incorrect Data Types
Ensure each column holds the correct data type for analysis.

> **📸 SCREENSHOT HERE** — Capture the before/after dtype comparison table.

In [3]:
# Check current dtypes before correction
print('Data types BEFORE correction:')
print(df.dtypes)
print()

Data types BEFORE correction:
flow_id                 str
src_ip                  str
src_port              int64
dst_ip                  str
dst_port              int64
protocol                str
timestamp               str
flow_duration       float64
tot_fwd_pkts        float64
tot_bwd_pkts        float64
pkt_size_avg        float64
fwd_iat_mean        float64
bwd_iat_mean        float64
flow_bytes_per_s    float64
flow_pkts_per_s     float64
flag                    str
label                   str
dtype: object



In [4]:
# --- Correct src_port and dst_port to integer (they should be whole numbers) ---
# First coerce to numeric (invalid entries become NaN), then cast to Int64 (nullable int)
df['src_port'] = pd.to_numeric(df['src_port'], errors='coerce').astype('Int64')
df['dst_port'] = pd.to_numeric(df['dst_port'], errors='coerce').astype('Int64')

# --- Correct tot_fwd_pkts and tot_bwd_pkts to integer ---
df['tot_fwd_pkts'] = pd.to_numeric(df['tot_fwd_pkts'], errors='coerce').astype('Int64')
df['tot_bwd_pkts'] = pd.to_numeric(df['tot_bwd_pkts'], errors='coerce').astype('Int64')

# --- Correct flow_duration, pkt_size_avg, fwd_iat_mean etc. to float ---
float_cols = ['flow_duration', 'pkt_size_avg', 'fwd_iat_mean', 'bwd_iat_mean',
              'flow_bytes_per_s', 'flow_pkts_per_s']
for col in float_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# --- Ensure categorical columns are stored as string (object) ---
for col in ['protocol', 'flag', 'label']:
    df[col] = df[col].astype(str).replace('nan', np.nan)

print('Data types AFTER correction:')
print(df.dtypes)

Data types AFTER correction:
flow_id                 str
src_ip                  str
src_port              Int64
dst_ip                  str
dst_port              Int64
protocol                str
timestamp               str
flow_duration       float64
tot_fwd_pkts          Int64
tot_bwd_pkts          Int64
pkt_size_avg        float64
fwd_iat_mean        float64
bwd_iat_mean        float64
flow_bytes_per_s    float64
flow_pkts_per_s     float64
flag                    str
label                   str
dtype: object


---
### Step 2.4 — Fix Timestamp Formatting Inconsistencies
The dataset contains timestamps in 3 different formats. We parse and standardize all of them.

> **📸 SCREENSHOT HERE** — Capture the sample of original timestamps vs the standardized output.

In [5]:
# Show sample of raw timestamp values before fixing
print('Sample raw timestamp values (first 15):')
print(df['timestamp'].head(15).tolist())

Sample raw timestamp values (first 15):
['2024-01-03 10:47:07', '2024-01-19 14:55:33', '05/01/2024 04:20', '2024-01-20 00:36:58', '01-29-2024 22:31:54', '01-22-2024 15:26:12', '18/01/2024 06:19', '2024-01-17 03:11:41', '2024-01-26 21:33:17', '01-23-2024 14:34:27', '2024-01-07 09:33:09', '2024-01-23 17:34:51', '23/01/2024 02:47', '23/01/2024 14:13', '2024-01-12 05:56:35']


In [6]:
# Parse timestamps using multiple possible formats
# pandas infer_datetime_format handles mixed formats gracefully
df['timestamp'] = pd.to_datetime(df['timestamp'], infer_datetime_format=True, errors='coerce')

# Confirm how many parsed successfully vs failed (NaT)
nat_count = df['timestamp'].isna().sum()
print(f'Timestamps successfully parsed : {len(df) - nat_count}')
print(f'Timestamps failed to parse (NaT): {nat_count}')

print('\nSample standardized timestamps (first 10):')
print(df['timestamp'].head(10).tolist())

TypeError: to_datetime() got an unexpected keyword argument 'infer_datetime_format'

---
### Step 2.5 — Fix Invalid Port Numbers
Valid port range is 0–65535. Ports outside this range are invalid and should be nullified.

> **📸 SCREENSHOT HERE** — Capture the invalid port summary before and after.

In [7]:
# Identify invalid port values before correction
invalid_src = df[~df['src_port'].between(0, 65535, inclusive='both') & df['src_port'].notna()]
invalid_dst = df[~df['dst_port'].between(0, 65535, inclusive='both') & df['dst_port'].notna()]

print(f'Invalid src_port values before fix: {invalid_src["src_port"].unique().tolist()}')
print(f'Invalid dst_port values before fix: {invalid_dst["dst_port"].unique().tolist()}')
print(f'Count of invalid src_ports: {len(invalid_src)}')
print(f'Count of invalid dst_ports: {len(invalid_dst)}')

Invalid src_port values before fix: [99999, -1]
Invalid dst_port values before fix: []
Count of invalid src_ports: 354
Count of invalid dst_ports: 0


In [8]:
# Replace invalid port values with NaN (to be handled in Task 3)
df.loc[~df['src_port'].between(0, 65535, inclusive='both'), 'src_port'] = pd.NA
df.loc[~df['dst_port'].between(0, 65535, inclusive='both'), 'dst_port'] = pd.NA

# Confirm fix
still_invalid_src = df[~df['src_port'].between(0, 65535, inclusive='both') & df['src_port'].notna()]
print(f'Invalid src_port values after fix: {len(still_invalid_src)}')
print('Port values are now within valid range (0–65535) or NaN.')

Invalid src_port values after fix: 0
Port values are now within valid range (0–65535) or NaN.


In [9]:
print('======= TASK 2 CORRECTIONS SUMMARY =======')
print(f'  Column names standardized      : {len(df.columns)} columns fixed')
print(f'  Data types corrected           : ports → Int64, floats → float64, timestamps → datetime64')
print(f'  Timestamp formats unified      : all converted to datetime64')
print(f'  Invalid port values nullified  : out-of-range ports set to NaN')
print('===========================================')

print('\nFinal column names and dtypes after Task 2:')
print(df.dtypes)

print('\nFirst 5 rows after structural correction:')
df.head()

======= TASK 2 CORRECTIONS SUMMARY =======
  Column names standardized      : 17 columns fixed
  Data types corrected           : ports → Int64, floats → float64, timestamps → datetime64
  Timestamp formats unified      : all converted to datetime64
  Invalid port values nullified  : out-of-range ports set to NaN

Final column names and dtypes after Task 2:
flow_id                 str
src_ip                  str
src_port              Int64
dst_ip                  str
dst_port              Int64
protocol                str
timestamp               str
flow_duration       float64
tot_fwd_pkts          Int64
tot_bwd_pkts          Int64
pkt_size_avg        float64
fwd_iat_mean        float64
bwd_iat_mean        float64
flow_bytes_per_s    float64
flow_pkts_per_s     float64
flag                    str
label                   str
dtype: object

First 5 rows after structural correction:


,flow_id,src_ip,src_port,dst_ip,dst_port,protocol,timestamp,flow_duration,tot_fwd_pkts,tot_bwd_pkts,pkt_size_avg,fwd_iat_mean,bwd_iat_mean,flow_bytes_per_s,flow_pkts_per_s,flag,label
0,flow_0000,7.140.125.58,<NA>,171.84.26.102,22,Tcp,2024-01-03 10:47:07,NaN,212,140,527.75,2154.51,2757.13,819791.08,103.97,ACK,PortScan
1,flow_0001,27.44.216.9,51191,161.156.119.110,8080,Tcp,2024-01-19 14:55:33,NaN,-5,<NA>,374.60,3870.88,1352.54,424592.44,708.53,SYN,benign
2,flow_0002,NaN,<NA>,168.46.48.1,443,HTTP,05/01/2024 04:20,NaN,-5,<NA>,609.74,1044.10,178.00,71656.80,837.41,FIN,BENIGN
3,flow_0003,130.13.101.184,<NA>,55.244.39.34,22,HTTP,2024-01-20 00:36:58,-999.00,<NA>,264,1463.45,374.99,3251.68,740429.12,518.97,FIN,ddos
4,flow_0004,140.214.112.115,51228,152.115.227.3,21,UDP,01-29-2024 22:31:54,9999999.00,<NA>,452,1213.28,3138.95,1517.09,221694.04,382.59,FIN,benign


---
### Step 2.7 — Save Intermediate Dataset

In [10]:
# Save the structurally corrected dataset for use in Task 3
df.to_csv('cybersecurity_task2.csv', index=False)
print('Saved: cybersecurity_task2.csv')

Saved: cybersecurity_task2.csv


---
## Task 2 Summary

| Correction | Action Taken |
|---|---|
| Column name whitespace | Stripped with `.str.strip()` |
| Column name casing | Lowercased with `.str.lower()` |
| Column name separators | Spaces/slashes replaced with underscores |
| Port columns dtype | Converted to nullable `Int64` |
| Packet/flow columns dtype | Converted to `float64` |
| Timestamp formats | Unified to `datetime64` via `pd.to_datetime()` |
| Invalid port values | Out-of-range values set to `NaN` |

> **Next Step:** Task 3 — Handling Missing Values